# Avaliar um modelo aberto na GPU gratuita do Kaggle

Roda o benchmark com um modelo servido pelo Ollama dentro do notebook (GPU T4 ou P100, 30 h/semana grátis).

Antes de rodar: **Settings → Accelerator → GPU** e **Settings → Internet → On**.

Só os itens públicos do repositório são usados; os privados nunca saem da sua máquina.

In [ ]:
MODELO = "qwen3:8b"  # qualquer modelo de https://ollama.com/library que caiba na GPU
REPOSITORIO = "https://github.com/vitoriaayres/avaliacao-llm-ptbr.git"
LIMITE_POR_CATEGORIA = 0  # 0 = todos os itens

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, urllib.request

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434")
        break
    except OSError:
        time.sleep(1)
!ollama pull {MODELO}

In [ ]:
!git clone --depth 1 {REPOSITORIO} projeto
%cd projeto
!pip install -q -e .

# Registra o modelo do Ollama na configuração local do notebook.
import yaml
caminho = "configuracao/modelos.yaml"
config = yaml.safe_load(open(caminho, encoding="utf-8"))
id_modelo = f"ollama/{MODELO}"
if id_modelo not in {m["id"] for m in config["modelos"]}:
    config["modelos"].append({"id": id_modelo, "nome": f"{MODELO} (Kaggle)"})
    yaml.safe_dump(config, open(caminho, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)

Os itens do ENEM não ficam no repositório (direitos dos textos de apoio). Para incluí-los, suba os PDFs do INEP como *Dataset* do Kaggle e rode `avaliador extrair-enem` apontando para `/kaggle/input/...` antes da próxima célula.

In [ ]:
nome = f"kaggle-{MODELO.replace(':', '-')}"
limite = f"--limite {LIMITE_POR_CATEGORIA}" if LIMITE_POR_CATEGORIA else ""
!avaliador validar
!avaliador executar -m {id_modelo} --nome {nome} {limite}
!avaliador pontuar {nome} --sem-juiz

In [ ]:
# Baixe este zip em Output e coloque a pasta em resultados/ do seu projeto local.
!cd resultados && zip -qr /kaggle/working/{nome}.zip {nome}
servidor.terminate()